
# 🎼 Hindustani Raga Classification — PCD + PCDD

**Task:** classify the raga of a Hindustani classical audio clip using
only two melodic features, following the classic PCD/PCDD approach:

- **PCD (Pitch Class Distribution):** how often each tonic-relative pitch
  class is used — captures *which notes* the raga favors.
- **PCDD (Pitch Class Distribution of Dyads):** how often each pair of
  *consecutive* notes occurs — captures melodic movement, not just note
  frequency. Notes are found by segmenting the pitch track at detected
  onsets (same idea as the reference: *"PCDDs were determined by using
  the detected onsets to segment the pitch-tracks into notes"*), each
  note gets one pitch-class label, and consecutive notes are paired into
  dyads.

No timbral (MFCC) features here on purpose — this keeps the model
focused purely on melodic content, which is what raga identity actually
lives in, and keeps the code short.

**Known limitation (same one the reference calls out):** onset-based
note segmentation can't reliably separate notes reached by a slide
(*meend*) rather than a clean attack — a slide gets split at whatever
energy changes happen to look like an onset, or not split at all. This
is a heuristic, not ground truth; it's good enough to be a useful
feature, not perfect note transcription.

**Evaluation**, mirroring the reference:
- **Stratified 10-fold CV** — the standard "same performances, held-out
  clips" number.
- **Leave-one-artist-out CV** — stands in for the reference's *"unseen"*
  evaluation (excerpts from performances/performers not seen in
  training). Expect this number to be noticeably lower than the CV
  number, same as the reference's 99% (CV) vs 75% (unseen) gap — that
  gap is real signal about generalization, not a bug.
- **Classifier: SVM** (as in the reference) with an RBF kernel.

**Structure:**
1. Config
2. Imports
3. Raga UUID → label mapping
4. File discovery
5. Feature extraction: tonic, note segmentation, PCD, PCDD
6. Build the feature table
7. EDA
8. Modeling — SVM, Stratified CV + Leave-One-Artist-Out CV


## 1. Config

In [ ]:

from dataclasses import dataclass

@dataclass(frozen=True)
class Config:
    # --- paths ---------------------------------------------------------
    audio_root: str = "/path/to/RagaDataset/Hindustani/audio"  # folder containing <uuid> subfolders
    features_csv: str = "features_pcd_pcdd.csv"                 # cache for extracted features

    # --- audio -----------------------------------------------------------
    sample_rate: int = 22050
    analysis_duration: float | None = None   # seconds per clip to analyze; None = full clip
    fmin_hz: float = 80.0                    # vocal/instrument pitch search range
    fmax_hz: float = 800.0
    n_pitch_classes: int = 12                # PCD/PCDD resolution (12 = standard chroma-like);
                                              # PCDD size grows as n_pitch_classes^2, so keep this modest

    # --- parallelism -----------------------------------------------------
    n_jobs: int = 4   # Kaggle CPU notebooks give you 4 cores — use them for extraction

    # --- modeling --------------------------------------------------------
    random_seed: int = 42

CFG = Config()


## 2. Imports

In [ ]:

# Cap BLAS/OpenMP thread pools *before* numpy/librosa import them.
# Without this, each parallel worker below can spawn its own internal
# thread pool, oversubscribing Kaggle's cores.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import re
import warnings

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(CFG.random_seed)



## 3. Raga UUID → label mapping

Each raga's audio lives in its own UUID-named folder. This dict is the
ground truth for labels.


In [ ]:

# Raga IDs for 300 Hindustani Classical Music clips (30 ragas x 10 songs)
ID_TO_RAGA = {
    'f7fddfc0-8c1d-4dd2-90d5-5d51a99d61f8 2': 'Rāgēśrī',
    'f6432fec-e9c2-4b09-9e73-c46086cbd8ea': 'Ahira bhairav',
    '290674e0-d94c-41c1-ad99-f65fa22a1447': 'Madhuvanti',
    '0b3bbf97-0ec3-41da-add4-722d87329ec3': 'Madhukauns',
    'a9ee554f-f146-43e9-b2d0-9bb31fd4b57d': 'Kēdār',
    'd9c603fa-875f-4b84-b851-c6a345427898': 'Ābhōgī',
    '1e7de02f-e77f-405a-a033-f31117aaf955': 'Bhairav',
    '3eb7ba30-4b94-432e-9618-875ee57e01ab': 'Mārvā',
    '46997b02-f09c-4969-8138-4e1861f61967': 'Śrī',
    '54c4214c-05b9-4acc-8f77-6d5786e43a2e': 'Mārūbihāg',
    'a99e07d5-20a0-467b-8dcd-aa5a095177fd': 'Lalit',
    '62b79291-73b0-4c77-a353-0f8bc6ed8362': 'Dēś',
    '118401e7-8de8-4d81-9d8e-8070889e3fa8': 'Darbāri',
    '7591faad-e68a-4550-b675-8082842c6056': 'Bāgēśrī',
    '93c73081-bdf8-4eca-b325-d736b71e9b4b': 'Khamāj',
    'ba3242d0-8d40-4d93-865e-d2e2497ea2a8': 'Gauḍ malhār',
    'ecd04ceb-b46c-47fc-9045-84ac9160e527': 'Basant',
    '6f13484e-6fdd-402d-baf3-3835835454d0': 'Tōḍī',
    'fa28470c-d413-44c7-94da-181f530cbfdd': 'Puriyā dhanaśrī',
    'f7fddfc0-8c1d-4dd2-90d5-5d51a99d61f8': 'Rāgēśrī',
    '40dbe1db-858f-4366-bef8-9076eb67340d': 'Śuddh sāraṅg',
    '1b05a564-059f-445b-b325-cf26318367e3': 'Miyān malhār',
    'b143adaa-f1a6-4de4-8985-a5bd35e96279': 'Bairāgi',
    '64e5fb9e-5569-4e80-8e6c-f543af9469c7': 'Mālkauns',
    '2ed9379f-14c9-49af-8e4d-f9b63e96801f': 'Alahaiyā bilāval',
    'e771a74d-545d-41d5-816b-43403a818b0c': 'Bhūp',
    '063ea5a0-23b1-4bb5-8537-3d924fe8ebb3': 'Jōg',
    'a7d98897-e2fe-4d75-b9dc-e5b4dc88e1c6': 'Bihāg',
    '48b37bed-e847-4882-8a01-5c721e07f07d': 'Yaman kalyāṇ',
    '595de771-fcc2-414c-9df6-ae4ffd898549': 'Hamsadhvāni',
    'dd59147d-8775-44ff-a36b-0d9f15b31319': 'Bilāsakhānī tōḍī',
}

print(f"Loaded {len(ID_TO_RAGA)} raga id mappings "
      f"({len(set(ID_TO_RAGA.values()))} unique raga labels)")



## 4. File discovery

Each `<uuid>` folder under `audio_root` corresponds to one raga.
`clean_uuid` strips Kaggle-style duplicate suffixes (`... 2`, `... (2)`)
so re-uploaded duplicate folders still resolve to the correct label.
Artist is inferred from the first sub-folder under each UUID folder, for
leave-one-artist-out CV later.


In [ ]:

AUDIO_EXTENSIONS = (".wav", ".mp3", ".flac", ".m4a")


def clean_uuid(raw_name: str) -> str:
    '''Normalize a folder name to its canonical uuid string.'''
    name = raw_name.strip()
    name = re.sub(r"\s*\(\d+\)$", "", name)   # trailing " (1)", " (2)"
    name = re.sub(r"\s+\d+$", "", name)       # trailing " 2", " 3"
    return name.lower()


def build_normalized_lookup(id_to_raga: dict) -> dict:
    '''Collapse duplicate-suffixed uuid keys into one clean lookup table.'''
    return {clean_uuid(k): v for k, v in id_to_raga.items()}


def find_audio_files_with_labels(root_dir: str, id_to_raga: dict) -> list[tuple[str, str, str]]:
    '''Walk `root_dir` and return (filepath, raga_label, artist) for every
    audio file whose parent uuid folder is present in `id_to_raga`.
    '''
    lookup = build_normalized_lookup(id_to_raga)
    items = []

    for uuid_folder in sorted(os.listdir(root_dir)):
        uuid_path = os.path.join(root_dir, uuid_folder)
        if not os.path.isdir(uuid_path):
            continue

        raga_label = lookup.get(clean_uuid(uuid_folder))
        if raga_label is None:
            print(f"WARNING: no raga mapping for uuid folder '{uuid_folder}', skipping")
            continue

        for dirpath, _, filenames in os.walk(uuid_path):
            rel_path = os.path.relpath(dirpath, uuid_path)
            artist = rel_path.split(os.sep)[0] if rel_path != "." else "unknown"

            for fname in filenames:
                if fname.lower().endswith(AUDIO_EXTENSIONS):
                    items.append((os.path.join(dirpath, fname), raga_label, artist))

    return items



## 5. Feature extraction

### 5a. Tonic estimation

Hindustani music is *relative*-pitch based — "Sa" isn't a fixed
frequency. Every feature below is meaningless unless normalized to the
tonic first.

Note: octave errors in this estimate don't actually matter — PCD/PCDD
fold everything to a single octave (`mod 1200` cents), and an octave
error is exactly a multiple-of-1200-cents shift, which folding cancels
out.


In [ ]:

def estimate_tonic(y: np.ndarray, sr: int, fmin: float = CFG.fmin_hz, fmax: float = CFG.fmax_hz) -> float:
    '''Estimate the tonic (Sa) frequency in Hz via pitch-histogram peak-picking.'''
    f0, voiced_flag, _ = librosa.pyin(y, fmin=fmin, fmax=fmax, sr=sr, frame_length=2048)

    voiced_f0 = f0[voiced_flag == True] if voiced_flag is not None else f0[~np.isnan(f0)]
    voiced_f0 = voiced_f0[~np.isnan(voiced_f0)]
    if len(voiced_f0) == 0:
        return 220.0  # fallback if the clip has no detectable pitch

    cents = 1200 * np.log2(voiced_f0 / fmin)
    cents_folded = np.mod(cents, 1200)

    hist, bin_edges = np.histogram(cents_folded, bins=120, range=(0, 1200))
    peak_cents = bin_edges[np.argmax(hist)]
    return float(fmin * (2 ** (peak_cents / 1200)))



### 5b. Note segmentation, PCD, and PCDD

Three small functions:
1. `extract_note_sequence` — segment the clip at detected onsets, and
   assign each resulting note one tonic-relative pitch class (its median
   pitch during that segment).
2. `compute_pcd` — histogram of note pitch classes.
3. `compute_pcdd` — histogram of consecutive note *pairs* (dyads),
   flattened to a 1D vector.


In [ ]:

def extract_note_sequence(y: np.ndarray, sr: int, tonic_hz: float,
                           n_pitch_classes: int = CFG.n_pitch_classes,
                           fmin: float = CFG.fmin_hz, fmax: float = 1000.0) -> list[int]:
    '''Segment audio into notes via onset detection, and return the ordered
    sequence of tonic-relative pitch-class labels (one per note).
    '''
    f0, voiced_flag, _ = librosa.pyin(y, fmin=fmin, fmax=fmax, sr=sr, frame_length=2048)
    times = librosa.times_like(f0, sr=sr)

    onset_times = librosa.onset.onset_detect(y=y, sr=sr, units="time")
    duration = times[-1] if len(times) else len(y) / sr
    boundaries = np.unique(np.concatenate([[0.0], onset_times, [duration]]))

    note_classes = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        mask = (times >= start) & (times < end)
        seg_voiced = f0[mask & (voiced_flag == True)] if voiced_flag is not None else f0[mask]
        seg_voiced = seg_voiced[~np.isnan(seg_voiced)]
        seg_voiced = seg_voiced[seg_voiced > 0]
        if len(seg_voiced) == 0:
            continue  # silent / unvoiced segment — skip, no note here

        note_pitch = np.median(seg_voiced)
        cents_rel = 1200 * np.log2(note_pitch / tonic_hz)
        pitch_class = int(np.mod(cents_rel, 1200) / (1200 / n_pitch_classes)) % n_pitch_classes
        note_classes.append(pitch_class)

    return note_classes


def compute_pcd(note_classes: list[int], n_pitch_classes: int = CFG.n_pitch_classes) -> np.ndarray:
    '''Normalized histogram of note pitch classes.'''
    hist = np.zeros(n_pitch_classes)
    for pc in note_classes:
        hist[pc] += 1
    return hist / hist.sum() if hist.sum() > 0 else hist


def compute_pcdd(note_classes: list[int], n_pitch_classes: int = CFG.n_pitch_classes) -> np.ndarray:
    '''Normalized, flattened histogram of consecutive note-pitch-class pairs (dyads).'''
    hist = np.zeros((n_pitch_classes, n_pitch_classes))
    for a, b in zip(note_classes[:-1], note_classes[1:]):
        hist[a, b] += 1
    flat = hist.flatten()
    return flat / flat.sum() if flat.sum() > 0 else flat



## 6. Build the feature table

Cached to `CFG.features_csv` — delete that file (or change the path) to
force a re-extraction.


In [ ]:

def process_file(filepath: str, raga_label: str, artist: str,
                  sr_target: int = CFG.sample_rate,
                  duration: float | None = CFG.analysis_duration) -> dict:
    '''Load one clip and compute its PCD + PCDD feature row.'''
    y, sr = librosa.load(filepath, sr=sr_target, duration=duration, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)

    tonic_hz = estimate_tonic(y, sr)
    notes = extract_note_sequence(y, sr, tonic_hz)
    pcd = compute_pcd(notes)
    pcdd = compute_pcdd(notes)

    row = {"filepath": filepath, "artist": artist, "raga": raga_label,
           "tonic_hz": tonic_hz, "n_notes": len(notes)}
    row.update({f"pcd_{i}": v for i, v in enumerate(pcd)})
    row.update({f"pcdd_{i}": v for i, v in enumerate(pcdd)})
    return row


def _safe_process_file(filepath: str, raga_label: str, artist: str):
    '''Wrap process_file so one bad audio file can\'t kill the whole pool —
    workers return None on failure instead of raising.'''
    try:
        return process_file(filepath, raga_label, artist)
    except Exception as e:
        print(f"FAILED  {filepath}: {e}")
        return None


def build_feature_table(root: str, id_to_raga: dict, out_csv: str,
                         n_jobs: int = CFG.n_jobs) -> pd.DataFrame:
    '''Extract PCD + PCDD features for every labeled audio file and cache to CSV.

    Parallelized with joblib across `n_jobs` threads. Deliberately uses
    backend="threading", not the default process-based "loky": librosa's
    pyin/onset_detect are numba-JIT-compiled internally, and spawning
    multiple *processes* that each try to JIT-compile / spin up their own
    BLAS threads at once is a known cause of worker crashes
    (TerminatedWorkerError / SIGSEGV). Threads share one process's memory
    and numba releases the GIL during its compiled loops, so this still
    parallelizes without that crash risk. Set n_jobs=1 for a plain,
    easy-to-debug sequential loop.
    '''
    labeled_files = find_audio_files_with_labels(root, id_to_raga)
    print(f"Found {len(labeled_files)} audio files with a known raga label under {root}")
    print(f"Extracting with n_jobs={n_jobs}...")

    results = Parallel(n_jobs=n_jobs, backend="threading")(
        delayed(_safe_process_file)(filepath, raga_label, artist)
        for filepath, raga_label, artist in tqdm(labeled_files, desc="Extracting features")
    )
    rows = [r for r in results if r is not None]

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    n_ragas = df["raga"].nunique() if len(df) else 0
    print(f"\nSaved {len(df)} feature rows to {out_csv}")
    print(f"Ragas represented: {n_ragas} / {len(set(id_to_raga.values()))} in the mapping")
    return df


In [ ]:
# Process one raga at a time and cache each raga's features to its own
# CSV in /kaggle/working. If the kernel dies or you stop partway, already
# -finished ragas are skipped on re-run — only the remaining ones get
# (re)extracted. At the end, all per-raga CSVs are concatenated.

FEATURES_DIR = "/kaggle/working/raga_features"
os.makedirs(FEATURES_DIR, exist_ok=True)

raga_names = list(dict.fromkeys(ID_TO_RAGA.values()))  # unique raga names, order preserved
print(f"{len(raga_names)} ragas to process")

raga_dfs = []
for i, raga_name in enumerate(raga_names):
    # filesystem-safe filename (raga names can contain diacritics/spaces)
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", raga_name).strip("_")
    raga_csv = os.path.join(FEATURES_DIR, f"{safe_name}.csv")

    raga_id_to_raga = {uid: name for uid, name in ID_TO_RAGA.items() if name == raga_name}

    if os.path.exists(raga_csv):
        print(f"[{i+1}/{len(raga_names)}] cached: {raga_name}")
        raga_df = pd.read_csv(raga_csv)
    else:
        print(f"[{i+1}/{len(raga_names)}] extracting: {raga_name}")
        raga_df = build_feature_table(CFG.audio_root, raga_id_to_raga, raga_csv)

    raga_dfs.append(raga_df)

features_df = pd.concat(raga_dfs, ignore_index=True)
features_df.to_csv(CFG.features_csv, index=False)
print(f"\nCombined: {len(features_df)} rows across {features_df['raga'].nunique()} ragas")
features_df.head()


## 7. Exploratory data analysis

In [ ]:

print(f"Total clips: {len(features_df)}")
print(f"Unique ragas: {features_df['raga'].nunique()}")
print(f"Unique artists: {features_df['artist'].nunique()}")
print(f"Median notes detected per clip: {features_df['n_notes'].median():.0f}")

fig, ax = plt.subplots(figsize=(10, 6))
features_df["raga"].value_counts().sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Number of clips")
ax.set_title("Clips per raga — check this is roughly balanced")
plt.tight_layout()
plt.show()


In [ ]:

pcd_cols = [c for c in features_df.columns if c.startswith("pcd_")]
pcdd_cols = [c for c in features_df.columns if c.startswith("pcdd_")]

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
example = features_df.sample(1, random_state=CFG.random_seed).iloc[0]

axes[0].bar(range(len(pcd_cols)), example[pcd_cols].values, color="darkslateblue")
axes[0].set_title(f"PCD — {example['raga']} ({example['artist']})")
axes[0].set_xlabel("Pitch class")

axes[1].bar(range(len(pcdd_cols)), example[pcdd_cols].values, color="darkorange", width=1.0)
axes[1].set_title("PCDD (flattened pitch-class-pair histogram)")
axes[1].set_xlabel("Dyad index (pitch_class_a * n_pitch_classes + pitch_class_b)")

plt.tight_layout()
plt.show()



## 8. Modeling — SVM, Stratified CV + Leave-One-Artist-Out CV

Feature vector = PCD concatenated with PCDD. Matches the reference's
setup (PCD + PCDD features, SVM classifier); the two CV strategies give
you both their "CV" number and a stand-in for their "unseen" number.


In [ ]:

def evaluate_svm(X: np.ndarray, y: np.ndarray, groups: np.ndarray) -> dict:
    '''Evaluate an RBF-kernel SVM under stratified 10-fold CV and
    leave-one-artist-out CV.
    '''
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf")),
    ])

    n_splits_strat = max(2, min(10, pd.Series(y).value_counts().min()))
    skf = StratifiedKFold(n_splits=n_splits_strat, shuffle=True, random_state=CFG.random_seed)
    strat_scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy")

    n_splits_group = min(len(set(groups)), 10)
    gkf = GroupKFold(n_splits=n_splits_group)
    group_scores = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring="accuracy")

    return {
        "stratified_cv_acc_mean": strat_scores.mean(),
        "stratified_cv_acc_std": strat_scores.std(),
        "leave_artist_out_acc_mean": group_scores.mean(),
        "leave_artist_out_acc_std": group_scores.std(),
    }


In [ ]:

X = features_df[pcd_cols + pcdd_cols].fillna(0.0).values
y = features_df["raga"].values
groups = features_df["artist"].values

results = evaluate_svm(X, y, groups)

print(f"Stratified 10-fold CV accuracy:   {results['stratified_cv_acc_mean']:.3f} "
      f"± {results['stratified_cv_acc_std']:.3f}")
print(f"Leave-one-artist-out CV accuracy: {results['leave_artist_out_acc_mean']:.3f} "
      f"± {results['leave_artist_out_acc_std']:.3f}")
print("\n(A large gap between these two — like the reference's 99% vs 75% — is expected")
print("and reflects real generalization difficulty to unseen performers, not a bug.)")



## Next steps

- Try `n_pitch_classes` other than 12 — e.g. a finer resolution captures
  more microtonal detail, but PCDD grows quadratically, so watch for
  overfitting with only ~300 clips.
- The onset-based note segmentation is the weakest link (same limitation
  the reference notes). If leave-one-artist-out accuracy is much lower
  than CV accuracy, that's partly a modeling-generalization issue and
  partly a note-segmentation-quality issue — worth inspecting `n_notes`
  per clip for outliers (too few/many detected notes).
- If you have Essentia available, swap `estimate_tonic` for
  `TonicIndianArtMusic` — purpose-built for Indian art music.
